<a href="https://colab.research.google.com/github/joehiggi/graph-theoretic-applications-in-insurance-risk-management/blob/main/notebooks/edgar_exposure_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Disclosure-Derived Exposure Networks from SEC 10-K Filings

This notebook mirrors the `edgar_exposure` pipeline end to end:
**load** filings from `PleIAs/SEC` &rarr; **split** into 10-K Item sections &rarr;
**build** the bipartite firm-exposure graph and its firm-firm projection &rarr;
**measure** centrality &rarr; **visualise** with Graphviz.

## 1. Setup

Graphviz is a system dependency required by `pygraphviz`. On Colab, install it first.

In [ ]:
# Colab only: install the system Graphviz binaries and Python deps.
!apt-get -qq install -y graphviz graphviz-dev
!pip install -q datasets networkx pygraphviz pandas

# Make the edgar_exposure package importable (clone if running on Colab).
import os
if not os.path.isdir('src/edgar_exposure'):
    !git clone -q https://github.com/joehiggi/graph-theoretic-applications-in-insurance-risk-management.git
    %cd graph-theoretic-applications-in-insurance-risk-management
!pip install -q -e .

## 2. Load filings from `PleIAs/SEC`

We stream a small, bounded sample so we never download the full corpus.

In [ ]:
from edgar_exposure.loader import load_filings_list

filings = load_filings_list(limit=200)
print(f'Loaded {len(filings)} filings')
print(filings[0]['company'], '-', filings[0]['text'][:200], '...')

## 3. Split a filing into Item sections

In [ ]:
from edgar_exposure.sections import split_sections, get_section

sections = split_sections(filings[0]['text'])
print('Detected items:', list(sections))
print(get_section(filings[0]['text'], 'Item 1A')[:400])

## 4. Build the exposure graphs

In [ ]:
from edgar_exposure.graph import build_exposure_graph, project_firm_network

bipartite = build_exposure_graph(filings)
firms = project_firm_network(bipartite)
print('Bipartite graph:', bipartite)
print('Firm projection:', firms)

## 5. Graph-theoretic metrics

In [ ]:
from edgar_exposure.metrics import firm_centrality, exposure_frequency, summary

print(summary(bipartite))
display(exposure_frequency(bipartite).head(10))
display(firm_centrality(firms).head(10))

## 6. Visualise with Graphviz

In [ ]:
from edgar_exposure.viz import draw_graph
from IPython.display import Image

out = draw_graph(bipartite, 'outputs/exposure_network.png')
Image(str(out))